# HDF5DB

用于将因子数据保存在本地文件中，支持读取、增量写入、元信息管理与表/因子变更。使用流程是 `connect() → getTable() → readData()`；持久化时调用 `writeData()` 或 `writeFactorData()`。

文件布局、数据类型与锁机制见 [原理和细节](原理和细节.ipynb#hdf5-storage)。

In [ ]:
import logging
import warnings
warnings.filterwarnings('ignore')
from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [ ]:
import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from QuantStudio.Core.QSObject import Panel
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Tools.Visualization import qs_help

## 本篇示例数据

下面在临时目录中创建演示行情，读写、改名和删除操作均作用于该临时库。真实应用将 `MainDir` 替换为使用者指定的目录。

# 创建与连接

HDF5DB 的核心构造参数：

| 参数 | 类型 | 说明 |
|------|------|------|
| `MainDir` | `pathlib.Path` | **必填**，存放数据的主目录路径 |
| `FileOpenRetryNum` | `int \| inf` | 打开文件失败时的重试次数，默认 `inf`（无限重试） |

默认配置文件路径为 `~/QuantStudioConfig/HDF5DBConfig.json`。

In [ ]:
# 在临时目录中创建独立的演示库，所有数据由固定随机种子生成。
demo_directory = TemporaryDirectory()
demo_root = Path(demo_directory.name)
factor_dir = demo_root / "HDF5"
factor_dir.mkdir()
FDB = HDF5DB(args={"MainDir": factor_dir}).connect()
rng = np.random.default_rng(0)
demo_dates = list(pd.date_range("2025-01-01", "2025-04-30").to_pydatetime())
demo_ids = [f"{i:06d}.SZ" for i in range(1, 16)]
demo_shape = (len(demo_dates), len(demo_ids))
demo_price = pd.DataFrame(
    10 * np.exp(np.cumsum(rng.normal(.001, .01, demo_shape), axis=0)),
    index=demo_dates, columns=demo_ids,
)
FDB.writeData(Panel({"close": demo_price, "open": demo_price * .999,
                     "high": demo_price * 1.01, "low": demo_price * .99}),
              table_name="stock_cn_day_bar")
FDB.writeData(Panel({"if_listed": pd.DataFrame(1., index=demo_dates, columns=demo_ids)}),
              table_name="stock_cn_status")
FDB.writeData(Panel({"industry": pd.DataFrame(
    np.tile(["TMT", "Ind", "Fin"] * 5, (len(demo_dates), 1)),
    index=demo_dates, columns=demo_ids)}),
    table_name="stock_cn_industry", data_type={"industry": "string"})
FDB.writeData(Panel({name: pd.DataFrame(rng.uniform(.01, .2, demo_shape),
                                      index=demo_dates, columns=demo_ids)
                    for name in ("bp_lr", "ep_ttm")}), table_name="stock_cn_factor_value")
FDB.writeData(Panel({"close": demo_price.mean(axis=1).to_frame("000905.SH")}),
              table_name="index_cn_day_bar")
print(FDB.TableNames)

In [ ]:
# 获取因子库中的因子表列表
print(FDB.TableNames)

# 因子表

`HDF5FactorTable` 继承自 `FactorTable`，在其基础上增加了 HDF5 特有的参数：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `LookBack` | `int \| inf` | `0` | 缺失填充最大回溯天数。`0` 表示不填充，`inf` 表示无限制回溯填充 |
| `OnlyStartLookBack` | `bool` | `False` | 为 `True` 时只对提取数据的第一个时点进行缺失填充 |
| `OnlyLookBackNontarget` | `bool` | `False` | 为 `True` 时只使用不在目标时点序列中的数据来填充缺失 |
| `OnlyLookBackDT` | `bool` | `False` | 为 `True` 时所有 ID 统一沿时点维度回溯填充，不按每个 ID 单独填充 |
| `TargetDT` | `datetime \| None` | `None` | 非 `None` 时忽略时点序列，只取该时点的值广播到所有请求时点 |

In [ ]:
# 获取因子表对象
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})

In [ ]:
print(qs_help(FT))

In [ ]:
# 获取因子表中的因子列表
print(FT.FactorNames)

## 缺失填充

`LookBack` 参数控制缺失值的回溯填充行为。HDF5 文件的优势在于可以高效地读取历史数据来做缺失填充。

In [ ]:
# LookBack=0：不填充缺失
FT = FDB.getTable("stock_cn_day_bar", args={"LookBack": 0})

DTs = [dt.datetime(2025, 1, 3), dt.datetime(2025, 1, 4), dt.datetime(2025, 1, 5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs).loc[:, DTs, IDs]
print("LookBack=0 :")
print(Data)

print("-" * 10)
# LookBack=2：最多回溯 2 天填充缺失
FT2 = FDB.getTable("stock_cn_day_bar", args={"LookBack": 2})
Data2 = FT2.readData(factor_names=["open", "close"], ids=IDs, dts=DTs).loc[:, DTs, IDs]
print("LookBack=2 :")
print(Data2)

## 元信息

In [ ]:
# 获取因子表元信息
print(FT.getMetaData())

In [ ]:
# 获取因子元信息（批量）
print(FT.getFactorMetaData(factor_names=["close", "open"]))

## 时点序列与 ID 序列

HDF5FactorTable 的 `getDateTime` 和 `getID` 方法直接从 HDF5 文件的 Dataset 中读取。

传入 `ifactor_name` 可以获取特定因子的时点/ID；传入 `idt`（对 getID）或 `iid`（对 getDateTime）可以进一步过滤到有数据的范围。

In [ ]:
# 获取时点序列
print(FT.getDateTime(start_dt=dt.datetime(2025, 3, 1), end_dt=dt.datetime(2025, 3, 5)))

In [ ]:
# 获取 ID 序列
IDs = FT.getID()
print(IDs[0], ", ..., ", IDs[-1], f"共 {len(IDs)} 个")

## 读取数据

数据的读取有两种方式：
- **因子表级别 `readData`**：返回 `Panel`（继承自 FactorTable），适用于批量读取多个因子
- **单因子级别 `readFactorData`**：返回 `DataFrame`（HDF5FactorTable 特有），直接读取单个 HDF5 文件，更高效

In [ ]:
# 因子表级别 readData：返回 Panel
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

In [ ]:
# 单因子级别 readFactorData：直接读 HDF5 文件
print(qs_help(FT.readFactorData))

In [ ]:
# 使用 readFactorData 读取单个因子的原始数据
Data = FT.readFactorData(ifactor_name="close", ids=IDs, dts=DTs)
print("单因子数据 (DataFrame):")
print(Data)

# 因子

通过 `FT.getFactor()` 获取的因子对象是通用的 `Factor` 实例，其 `readData`、`getID`、`getDateTime` 等 API 与其他因子库中的因子完全一致。详细说明请参见 **[基本框架](基本框架.ipynb)** 和 **[因子开发](因子开发.ipynb)**。

In [ ]:
# 获取因子对象
F = FT.getFactor("close")
print(F.getMetaData(key=None))

In [ ]:
# 因子读取数据
Data = F.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs)
print(Data)

# 因子库的变更操作

HDF5DB 继承自 `WritableFactorDB`，支持完整的写入和变更操作。以下操作均受锁机制保护。

## 数据写入

`writeData(Panel, table_name=...)` 批量写入，`writeFactorData(DataFrame, table_name=..., ifactor_name=...)` 写入单个因子。

| `if_exists` | 行为 |
|---|---|
| `"update"` | 覆盖本次指定的日期与证券位置，允许新值为空；未指定的位置保留 |
| `"append"` | 只填充旧数据为空的位置 |
| `"update_notnull"` | 只用本次非空值覆盖旧值 |

当前 HDF5DB 未实现 `"replace"` 写入模式。若要清空后重建，先显式删除目标因子或表，再写入。

In [ ]:
# 创建测试数据
from QuantStudio.Core.QSObject import Panel

IDs = [str(i).zfill(6) + ".SZ" for i in range(1, 4)]
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]

Data = Panel({
    "Factor1": pd.DataFrame(rng.normal(size=(5, 3)), index=DTs, columns=IDs),
    "Factor2": pd.DataFrame(rng.normal(size=(5, 3)), index=DTs, columns=IDs)
})
print("待写入的数据 : ")
print(Data)

In [ ]:
# 使用 writeData 批量写入
TargetTable = "TestTable"
FDB.writeData(data=Data, table_name=TargetTable, if_exists="update", data_type={"Factor1":"double", "Factor2":"double"})

print("写入后的因子表列表 : ")
print(FDB.TableNames)
print()
print("写入后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

In [ ]:
# 使用 writeFactorData 追加单个因子
NewFactor = pd.DataFrame(rng.normal(size=(5, 3)), index=DTs, columns=IDs)
FDB.writeFactorData(factor_data=NewFactor, table_name=TargetTable, ifactor_name="Factor3", if_exists="update", data_type="double")

print("追加后的因子列表 : ")
print(FDB.getTable(TargetTable).FactorNames)

## 元信息管理

In [ ]:
# 设置因子的元信息（支持单个 key 或批量 meta_data）
FDB.setFactorMetaData(table_name=TargetTable, ifactor_name="Factor1", key="Description", value="这是一个测试因子")

FT = FDB.getTable(TargetTable)
print("设置后的元信息 : ")
print(FT.getFactorMetaData(factor_names=["Factor1"]))

In [ ]:
# 设置因子表的元信息
FDB.setTableMetaData(table_name=TargetTable, meta_data={"Description": "这是一张测试表", "Version": 1})

print("设置后的表元信息 : ")
print(FT.getMetaData())

## 重命名与删除

In [ ]:
# 重命名因子
print("重命名前 : ", FT.FactorNames)
FDB.renameFactor(table_name=TargetTable, old_factor_name="Factor1", new_factor_name="NewFactor1")
print("重命名后 : ", FT.FactorNames)

In [ ]:
# 删除因子
FDB.deleteFactor(table_name=TargetTable, factor_names=["NewFactor1"])
print("删除后 : ", FT.FactorNames)

In [ ]:
# 重命名因子表
print("重命名前 : ", FDB.TableNames)
FDB.renameTable(old_table_name=TargetTable, new_table_name="NewTestTable")
print("重命名后 : ", FDB.TableNames)

In [ ]:
# 删除因子表
FDB.deleteTable(table_name="NewTestTable")
print("删除后 : ", FDB.TableNames)

# 数据维护

HDF5DB 提供了两个数据维护工具方法：

| 方法 | 说明 |
|------|------|
| `optimizeData(table_name, factor_names)` | 对指定因子按 DateTime 排序以优化存储。当大量追加写入导致数据乱序后，可调用此方法整理 |
| `fixData(table_name, factor_names)` | 修复数据不一致问题：ID/Datetime 长度与 Data 不匹配、ID/Datetime 重复值等 |